# Phase 2 - B8 Function-vector-guided replay adaptation (Kaggle)

This notebook runs a documented B8 activation-vector adaptation on the frozen five-language XNLI protocol using the shared Phase 1 candidate-label prompt and LoRA trainer. After each stage it stores a normalized mean final-layer prompt activation per language. Later stages replay a uniform 125-example prior-language reservoir and add a cosine activation-anchor loss on replay prompts. This is not the source paper's activation-patched attention-head function vector or function-vector-guided KL objective (Jiang et al., 2025, arXiv:2502.11019).

All five orders run by default. To fit multiple Kaggle sessions, change `ORDER_IDS` to run one or two orders, save a notebook version with output, then attach that saved output as input to the next session and set `PREVIOUS_ARTIFACTS_DIR` to its matching artifacts directory. Completed orders are validated and skipped; an interrupted order restarts from that order's beginning.

## Before running
1. Enable a Kaggle GPU and Internet.
2. Push the B8 runner changes to the public repository, since this notebook clones the repository before execution.
3. Accept the gated model terms for `meta-llama/Llama-3.2-1B` and add a Kaggle secret named `HF_TOKEN`.
4. Run all cells, then use **Save Version** with notebook output enabled to preserve `/kaggle/working/phase2_b8_artifacts`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if REPO_ROOT.exists():
    raise FileExistsError(
        f"Repository directory already exists: {REPO_ROOT}. Restart the Kaggle session or choose a fresh path."
    )
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
repo_revision = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
print("Cloned repository revision:", repo_revision)

requirements_file = REPO_ROOT / "phase2" / "requirements-phase2.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Requirements file not found: {requirements_file}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)],
    check=True,
)
sys.path.insert(0, str(REPO_ROOT))

runner_path = REPO_ROOT / "phase2" / "run_baseline.py"
if not runner_path.is_file():
    raise FileNotFoundError(f"Missing B8 runner: {runner_path}")
help_result = subprocess.run(
    [sys.executable, "-m", "phase2.run_baseline", "--help"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(help_result.stdout)
if help_result.returncode != 0 or "B8" not in help_result.stdout:
    raise RuntimeError(
        "The cloned repository does not expose B8. Push the B8 runner changes to the default branch, then restart this Kaggle session."
    )

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

PROTOCOL_PATH = REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"
ARTIFACT_ROOT = Path("/kaggle/working/phase2_b8_artifacts")
PREVIOUS_ARTIFACTS_DIR = None  # Set to the mounted prior output's phase2_b8_artifacts directory to continue.
print({
    "repo": str(REPO_ROOT),
    "revision": repo_revision,
    "artifact_root": str(ARTIFACT_ROOT),
    "runner_file": str(runner_path),
})


In [ ]:
from phase2.preflight import collect_preflight
from phase2.prepare_data import prepare_xnli
from phase2.protocol import load_protocol
from phase2.run_baseline import validate_prepared_artifacts
import json

preflight = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(preflight, indent=2, sort_keys=True))
protocol = load_protocol(PROTOCOL_PATH)

if not (ARTIFACT_ROOT / "protocol_manifest.json").is_file():
    if ARTIFACT_ROOT.exists():
        raise FileExistsError(
            "A partial working artifact directory exists without a protocol manifest. Inspect it before continuing."
        )
    if PREVIOUS_ARTIFACTS_DIR is not None:
        previous = Path(PREVIOUS_ARTIFACTS_DIR)
        if not (previous / "protocol_manifest.json").is_file():
            raise FileNotFoundError(f"Previous artifact manifest not found under {previous}")
        shutil.copytree(previous, ARTIFACT_ROOT)
        print("Copied previous Kaggle B8 artifacts from:", previous)
    else:
        prepare_xnli(PROTOCOL_PATH, ARTIFACT_ROOT)
        print("Prepared the fixed XNLI subsets.")
prepared_manifest = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)

ALL_ORDER_IDS = [order["id"] for order in protocol["orders"]]
ORDER_IDS = ALL_ORDER_IDS.copy()  # For separate sessions, select one or two order IDs here.
if len(ALL_ORDER_IDS) != 5:
    raise ValueError(f"B8 expects five protocol orders; found {len(ALL_ORDER_IDS)}.")
if not ORDER_IDS or len(set(ORDER_IDS)) != len(ORDER_IDS) or set(ORDER_IDS) - set(ALL_ORDER_IDS):
    raise ValueError(f"Choose unique protocol order IDs from {sorted(ALL_ORDER_IDS)}.")
print({
    "languages": protocol["dataset"]["languages"],
    "orders_to_run": ORDER_IDS,
    "buffer_capacity": protocol["replay"]["buffer_capacity_examples"],
    "data_manifest_sha256": prepared_manifest["data_manifest_sha256"],
})


In [ ]:
command = [
    sys.executable,
    "-m",
    "phase2.run_baseline",
    "--method",
    "B8",
    "--protocol",
    str(PROTOCOL_PATH),
    "--artifact-root",
    str(ARTIFACT_ROOT),
]
for order_id in ORDER_IDS:
    command.extend(["--order-id", order_id])
print("RUN COMMAND:")
print(command)
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
from phase2.protocol import build_manifest
from phase2.results import validate_result_record
import json
import statistics

run_manifest = build_manifest(protocol)
run_manifest["data_artifacts"] = prepared_manifest["data_artifacts"]
run_manifest["data_manifest_sha256"] = prepared_manifest["data_manifest_sha256"]
expected_runs = {
    run["run_id"]: run
    for run in run_manifest["runs"]
    if run["method_id"] == "B8"
}
if len(expected_runs) != 5:
    raise ValueError(f"Expected five B8 runs in the current manifest; found {len(expected_runs)}.")
completed = {}
for run_id, run in expected_runs.items():
    result_path = ARTIFACT_ROOT / "results" / f"{run_id}.json"
    if result_path.is_file():
        record = json.loads(result_path.read_text(encoding="utf-8"))
        validate_result_record(record, run_manifest)
        completed[run["order_id"]] = record

metric_names = ("ACC", "BWT", "FWT", "F")
summary = {
    "method_id": "B8",
    "method_interpretation": "Function-vector-guided activation-anchor replay adaptation",
    "protocol_id": run_manifest["protocol_id"],
    "protocol_sha256": run_manifest["protocol_sha256"],
    "data_manifest_sha256": run_manifest["data_manifest_sha256"],
    "completed_orders": len(completed),
    "expected_orders": len(expected_runs),
    "complete": len(completed) == len(expected_runs),
    "per_order": {
        order_id: {name: record["metrics"][name] for name in metric_names}
        for order_id, record in sorted(completed.items())
    },
}
if completed:
    summary["across_order_summary"] = {
        name: {
            "mean": statistics.mean(record["metrics"][name] for record in completed.values()),
            "population_std": statistics.pstdev(record["metrics"][name] for record in completed.values()),
        }
        for name in metric_names
    }
summary_path = ARTIFACT_ROOT / "B8_summary.json"
temporary_path = summary_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
temporary_path.replace(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
print(f"Validated {len(completed)}/{len(expected_runs)} B8 orders. Save the Kaggle notebook version/output to preserve artifacts:", ARTIFACT_ROOT)


## Saved outputs and continuing in another Kaggle session

- `/kaggle/working/phase2_b8_artifacts/protocol_manifest.json` and `data/`: fixed selected examples and checksums.
- `phase2_b8_artifacts/checkpoints/<run_id>/`: adapters and tokenizers after each language stage.
- `phase2_b8_artifacts/ledgers/<run_id>/`: evaluation matrix and ledger.
- `phase2_b8_artifacts/results/<run_id>.json`: validated result for each completed order, including replay counts and activation-vector settings.
- `phase2_b8_artifacts/B8_summary.json`: validated results and summary; `complete` is true only after all five orders.

To continue in a new Kaggle session, attach the previous notebook output as an input, set `PREVIOUS_ARTIFACTS_DIR` to its mounted `phase2_b8_artifacts` directory, choose the remaining `ORDER_IDS`, then save the new output again. Keep the same data manifest and code revision for every order. The run summary is incomplete until all five orders validate; partial outputs can still be downloaded for continuation. Never tune using test scores.

In [ ]:
import json
import os
import shutil
from IPython.display import FileLink, display

summary_path = ARTIFACT_ROOT / "B8_summary.json"
if not summary_path.is_file():
    raise FileNotFoundError(f"B8 summary not found: {summary_path}")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
if summary.get("method_id") != "B8" or summary.get("expected_orders") != 5:
    raise RuntimeError("The summary does not match this five-order B8 protocol.")

os.chdir("/kaggle/working")
archive_path = Path(
    shutil.make_archive(
        "phase2_b8_artifacts",
        "zip",
        root_dir=ARTIFACT_ROOT.parent,
        base_dir=ARTIFACT_ROOT.name,
    )
)
if not archive_path.is_file() or archive_path.stat().st_size == 0:
    raise RuntimeError(f"B8 ZIP creation failed: {archive_path}")
print(f"Created {archive_path} ({archive_path.stat().st_size / (1024 ** 2):.1f} MiB)")
print("Download the B8 output ZIP; its summary states whether all five orders are complete:")
display(FileLink(archive_path.name, result_html_prefix="Download: "))
